# Model Comparison — XGBoost vs LSTM

Compares four model variants across metrics, ROC curves, feature importance, and sentiment uplift:
1. **XGBoost + sentiment**
2. **XGBoost (price-only)**
3. **LSTM + sentiment**
4. **LSTM (price-only)**

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

EVAL_PATH = Path("../evaluation/btc_usd_evaluation.json")
SAVED_DIR = Path("../models/saved")

with open(EVAL_PATH) as f:
    eval_data = json.load(f)

model_names = list(eval_data.keys())
print(f"Models loaded: {model_names}")

## 1. Metrics Comparison Table

In [ ]:
rows = []
for name, data in eval_data.items():
    m = data["metrics"]
    has_sentiment = "no_sentiment" not in name
    model_type = "XGBoost" if "xgb" in name else "LSTM"
    rows.append({
        "Model": model_type,
        "Sentiment": "Yes" if has_sentiment else "No",
        "F1": m["f1"],
        "Accuracy": m["accuracy"],
        "AUC-ROC": m["auc_roc"],
    })

metrics_df = pd.DataFrame(rows)
print(metrics_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metric_names = ["F1", "Accuracy", "AUC-ROC"]
colors = {"Yes": "#3498db", "No": "#e74c3c"}

for ax, metric in zip(axes, metric_names):
    x = np.arange(2)
    width = 0.35
    for i, sent in enumerate(["Yes", "No"]):
        vals = metrics_df[metrics_df["Sentiment"] == sent][metric].values
        ax.bar(x + i * width, vals, width, label=f"Sentiment: {sent}",
               color=colors[sent], edgecolor="white")
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(["XGBoost", "LSTM"])
    ax.set_title(metric, fontsize=13, fontweight="bold")
    ax.legend()
    ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5)

plt.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. ROC Curves

AUC-ROC measures how well the model ranks predictions — a more robust metric than accuracy for binary classification.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

styles = {
    "xgb_all_features": ("#2c3e50", "-", "XGBoost + Sentiment"),
    "xgb_no_sentiment": ("#2c3e50", "--", "XGBoost (price-only)"),
    "lstm_all_features": ("#e67e22", "-", "LSTM + Sentiment"),
    "lstm_no_sentiment": ("#e67e22", "--", "LSTM (price-only)"),
}

for name, data in eval_data.items():
    color, ls, label = styles[name]
    auc = data["metrics"]["auc_roc"]
    ax.plot(data["fpr"], data["tpr"], color=color, linestyle=ls, linewidth=2,
            label=f"{label} (AUC={auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — All Models", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=11)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

## 3. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
labels_cm = ["Down", "Up"]

for ax, (name, data) in zip(axes, eval_data.items()):
    cm = np.array(data["metrics"]["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels_cm,
                yticklabels=labels_cm, ax=ax, cbar=False)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    short_name = name.replace("_", " ").replace("all features", "+ Sent.").replace("no sentiment", "No Sent.")
    ax.set_title(short_name, fontsize=11)

plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 4. XGBoost Feature Importance

Which features does the tree model rely on most? Sentiment features appearing in the top ranks validates the project's hypothesis.

In [ ]:
fi = pd.read_csv(SAVED_DIR / "btc_usd_xgb_all_features_feature_importance.csv")

sentiment_keywords = ["fng", "news", "sentiment", "extreme", "divergence", "volume_fng"]
fi["is_sentiment"] = fi["feature"].apply(
    lambda f: any(kw in f for kw in sentiment_keywords)
)

colors_fi = ["#e67e22" if s else "#3498db" for s in fi["is_sentiment"]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(fi["feature"][::-1], fi["importance"][::-1], color=colors_fi[::-1], edgecolor="white")
ax.set_xlabel("Importance (Gain)")
ax.set_title("XGBoost Feature Importance (All Features)", fontsize=13, fontweight="bold")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#e67e22", label="Sentiment feature"),
    Patch(color="#3498db", label="Price/technical feature"),
], loc="lower right")

plt.tight_layout()
plt.show()

sent_pct = fi[fi["is_sentiment"]]["importance"].sum() / fi["importance"].sum() * 100
print(f"Sentiment features account for {sent_pct:.1f}% of total feature importance")

## 5. Sentiment Uplift Summary

The key question: does adding sentiment features measurably improve predictions?

In [ ]:
uplift_data = []
for model_type, prefix in [("XGBoost", "xgb"), ("LSTM", "lstm")]:
    with_m = eval_data[f"{prefix}_all_features"]["metrics"]
    without_m = eval_data[f"{prefix}_no_sentiment"]["metrics"]
    for metric in ["f1", "accuracy", "auc_roc"]:
        uplift_data.append({
            "Model": model_type,
            "Metric": metric.upper().replace("_", "-"),
            "With Sentiment": with_m[metric],
            "Without Sentiment": without_m[metric],
            "Uplift": with_m[metric] - without_m[metric],
        })

uplift_df = pd.DataFrame(uplift_data)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(uplift_df))
colors_uplift = ["#2ecc71" if u > 0 else "#e74c3c" for u in uplift_df["Uplift"]]
bars = ax.bar(x, uplift_df["Uplift"], color=colors_uplift, edgecolor="white")

labels_x = [f"{row['Model']}\n{row['Metric']}" for _, row in uplift_df.iterrows()]
ax.set_xticks(x)
ax.set_xticklabels(labels_x, fontsize=10)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_ylabel("Uplift (with − without sentiment)")
ax.set_title("Sentiment Feature Uplift Across Models & Metrics", fontsize=13, fontweight="bold")

for bar, val in zip(bars, uplift_df["Uplift"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{val:+.4f}", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)

plt.tight_layout()
plt.show()

print("\nDetailed uplift table:")
print(uplift_df.to_string(index=False))